# WearTwin AI demo
Sensor-style windows go in; the soldier's **digital-twin state** and **readiness score (0-100)** come out.

**All training data is synthetic** (generated by `simulate.py`). This shows the pipeline works. It is not evidence of accuracy on real soldiers.

Run the cells top to bottom. **Step 2 takes about 5 minutes. Do it before the presentation, not during it.**

## 1. Upload the project

In [ ]:
# A file picker appears: choose weartwin_demo.zip
from google.colab import files
files.upload()
!unzip -oq weartwin_demo.zip

import glob, os
os.chdir(os.path.dirname(glob.glob('**/train_eval.py', recursive=True)[0]))
print("working folder:", os.getcwd())
print(sorted(os.listdir('.')))

## 2. Train the model here (about 5 minutes)
Retraining in Colab avoids a mismatch between the library versions that saved the shipped model and the ones Colab has.

In [ ]:
!python train_eval.py 2>&1 | tail -n 22
!python make_figures.py 2>&1 | tail -n 3

## 3. Six hand-made scenarios
Each scenario is 10 minutes of resting calibration, then values that move gradually to the END values defined in `make_inputs.py`.

In [ ]:
import matplotlib.pyplot as plt, pandas as pd
from IPython.display import display
from engine import load_bundle
from make_inputs import SCENARIOS, run
from plot_scenarios import plot_scenarios

bundle = load_bundle()
logs = {name: run(name, n, s, e, bundle) for name, (n, s, e) in SCENARIOS.items()}
rows = []
for name, log in logs.items():
    live = [o for o in log if o["status"] == "Live"]
    rows.append(dict(scenario=name, final_state=live[-1]["state"],
                     final_readiness=round(live[-1]["readiness"]), band=live[-1]["band"]))
display(pd.DataFrame(rows))
plot_scenarios(logs)
plt.show()

## 4. Feed your own values
Move the sliders, then run the cell. The twin sees 10 minutes of rest, then values rising smoothly from rest to the ones you set over the chosen duration.

In [ ]:
#@title Your input { display-mode: "form" }
hr = 140 #@param {type:"slider", min:50, max:200, step:1}
rmssd = 10 #@param {type:"slider", min:3, max:100, step:1}
spo2 = 97 #@param {type:"slider", min:70, max:100, step:1}
rr = 28 #@param {type:"slider", min:6, max:55, step:1}
skin_t = 37.0 #@param {type:"slider", min:28, max:39, step:0.1}
amb_t = 34 #@param {type:"slider", min:-10, max:45, step:1}
hum = 50 #@param {type:"slider", min:10, max:100, step:1}
imu = 0.25 #@param {type:"slider", min:0, max:1, step:0.01}
minutes = 60 #@param {type:"slider", min:10, max:180, step:5}

import matplotlib.pyplot as plt
from engine import load_bundle
from make_inputs import REST, run
from plot_scenarios import plot_single

end = dict(hr=hr, rmssd=rmssd, spo2=spo2, rr=rr, skin_t=skin_t, amb_t=amb_t, hum=hum, imu=imu)
log = run("custom", int(minutes * 2), REST, end, load_bundle())
last = [o for o in log if o["status"] == "Live"][-1]
print(f"Final: {last['state']} | readiness {last['readiness']:.0f} ({last['band']}) | "
      f"thermal {last['thermal_idx']:.2f}  fatigue {last['fatigue_idx']:.2f}  cardio {last['cardio_idx']:.2f}")
plot_single(log, "Your input: rest -> " + ", ".join(f"{k} {v}" for k, v in end.items()))
plt.show()

## 5. Six unseen soldiers replayed as a live feed
Ground truth (from the simulator) is shown next to the twin's output.

In [ ]:
!python demo_replay.py --print-every 1000 | tail -n 22
from IPython.display import Image, display
display(Image('reports/fig_squad_replay.png'))

## 6. How good is it on held-out soldiers?

In [ ]:
from IPython.display import Image, display
display(Image('reports/fig_confusion.png'))